# Jupyter Notebook: SLC9000 API #

## Import modules and define connection settings

This cell sets up the environment for talking to the SLC9000 device. It imports required Python libraries, reads credentials from environment variables, configures the `requests` session (TLS verification, headers, timeout), and defines the base URL and other constants used by the REST API calls.


In [1]:
import json
import os
import requests
import time
from pprint import pprint

TIMEOUT = 10
VERIFY = False

# Create & use these environment variables or the load_config function
# USERNAME = os.environ.get("SLC9K_USER", "sysadmin")
USERNAME = ""
PASSWORD = ""
# PASSWORD = os.environ.get("SLC9K_PW", "")
PERCEPXION_HOST = ""
#
requests.packages.urllib3.disable_warnings()
session = requests.Session()
session.verify = VERIFY
session.timeout = TIMEOUT  # this apparently does nothing
session.headers.update(
    {
        "Content-Type": "application/json",
        "Accept": "application/json",
   }
)
session.base_url = "https://10.40.21.41"  # booth: slc9000-rh-dvt3

### Load configuration from JSON file

In [2]:
def load_config(filename="config.json"):
    """
    Load connection and authentication settings from a JSON config file.

    This function reads the given JSON configuration file and updates the
    global TIMEOUT, VERIFY, USERNAME, and PASSWORD variables. It allows the
    SLC9000 API demo to be configured without hard‑coding values in the
    notebook or relying solely on environment variables.

    The expected JSON structure is:
        {
            "TIMEOUT": 10,
            "VERIFY": false,
            "USERNAME": "sysadmin",
            "PASSWORD": "ciscolive"
        }

    Args:
        filename: Path to the JSON configuration file to load. Defaults to
            "config.json" in the current working directory.
    """
    global TIMEOUT, VERIFY, USERNAME, PASSWORD, PERCEPXION_HOST
    with open(filename, "r") as f:
        config = json.load(f)

    TIMEOUT = config.get("TIMEOUT", TIMEOUT)
    VERIFY = config.get("VERIFY", VERIFY)
    USERNAME = config.get("USERNAME", USERNAME)
    PASSWORD = config.get("PASSWORD", PASSWORD)
    PERCEPXION_HOST = config.get("PERCEPXION_URL", PERCEPXION_HOST)


## Helper functions for SLC9000 REST API calls

In [3]:
def api_call(
    method: str,
    path: str,
    payload: dict | None = None
):
    """
    Perform an HTTP request to the SLC9000 API using the shared session.

    This helper builds a full URL from the session's base_url and the given
    path, sends the request with the specified HTTP method, and handles basic
    error reporting. On success it returns the underlying requests.Response
    object; on failure it logs the error and returns None.

    Args:
        method: HTTP method to use, e.g. "GET", "POST", "PUT", or "DELETE".
        path: API path to append to session.base_url, e.g. "/api/v2/system/status".
        payload: Optional JSON-serializable dictionary to send as the request body
            for POST, PUT, and DELETE requests.

    Returns:
        A requests.Response object if the request succeeds; otherwise None.

    Raises:
        ValueError: If an unsupported HTTP method is provided.
    """
    url = f"{session.base_url}{path}"
    try:
        if method == "GET":
            response = session.get(url, timeout=TIMEOUT)
        elif method == "POST":
            response = session.post(url, json=payload or {}, timeout=TIMEOUT)
        elif method == "PUT":
            response = session.put(url, json=payload or {}, timeout=TIMEOUT)
        elif method == "DELETE":
            response = session.delete(url, json=payload or {}, timeout=TIMEOUT)
        else:
            raise ValueError(f"Unsupported method: {method}")

        response.raise_for_status()
        print(f"{url} successful ({response.status_code})")
        return response
    except requests.exceptions.HTTPError as http_err:
        body = response.text if "response" in locals() else "<no response>"
        print(f"HTTP error: {http_err} - Response: {body}")
    except requests.exceptions.RequestException as err:
        print(f"Request error: {err}")
    return None


# User Management
def user_login(username: str, password: str):
    """
    User login for all roles
    """
    response = api_call(
        "POST",
        "/api/v2/user/login",
        {"username": username, "password": password},
    )

    if response is None:
        return None

    data = response.json()
    token = data.get("token")
    if token:
        session.headers.update({"X-auth-token": token})
    print(token)
    return response

def sessions():
    """
    Get active sessions
    """
    return api_call("GET", "/api/v2/sessions")

def sessions_delete(session_id: int | str):
    """
    Terminate a session
    """
    return api_call("DELETE", f"/api/v2/sessions/{session_id}")

def user_logout():
    """
    Logout from an API session
    """
    return api_call("DELETE", "/api/v2/user/login")


# System
def system_identity():
    """
    Get basic system info
    """
    return api_call("GET", "/api/v2/system/identity")

def system_reboot():
    """
    Reboot system
    """
    return api_call("POST", "/api/v2/system/reboot")

def system_status():
    """
    Get system status
    """
    return api_call("GET", "/api/v2/system/status")

def system_version():
    """
    Get HW/SW versions
    """
    return api_call("GET", "/api/v2/system/version")

def system_ztp():
    """
    Get ZTP/bootstrap Status
    """
    return api_call("GET", "/api/v2/system/ztp")


# Network
def network_interfaces():
    """
    Get basic network info
    """
    return api_call("GET", "/api/v2/network/interfaces")

def network_interfaces_set(payload: dict | None = None):
    """
    Get basic network info
    """
    return api_call("PUT", "/api/v2/network/interfaces", payload=payload)


# Firmware
def firmware_version():
    """
    Get Firmware Version
    """
    return api_call("GET", "/api/v2/firmware/version")

def firmware_bootbank():
    """
    Get current boot bank
    """
    return api_call("GET", "/api/v2/firmware/bootbank")

def firmware_check():
    """
    Check for firmware updates
    """
    return api_call("GET", "/api/v2/firmware/check")

def firmware_bootbank_set(payload: dict | None = None):
    """
    Set active boot bank
    Example payload:
      {"bank": 1}
    """
    return api_call("PUT", "/api/v2/firmware/bootbank", payload=payload)

def firmware_log():
    """
    Get FW Update Log
    """
    return api_call("GET", "/api/v2/firmware/log")


# Config
def config_commands():
    """
    Get config (CLI commands)
    """
    return api_call("GET", "/api/v2/config/commands")

### Initialize global variables

In [4]:
load_config()

print(f"TIMEOUT={TIMEOUT}")
print(f"VERIFY={VERIFY}")

TIMEOUT=10
VERIFY=False


## Authenticate and obtain API session token

This section logs in to the SLC9000 using the provided username and password. On success, the device returns a JSON response containing a session `token`, its `expires_in` lifetime (in seconds), and user details. The token is used by the REST API to identify and authorize subsequent requests made during this session.

In [5]:
login = user_login(username=USERNAME, password=PASSWORD)

if login is not None:
    pprint(login.json())

https://10.40.21.41/api/v2/user/login successful (200)
bf75b888-d08a-49b2-a714-01bed8647e6e
{'authenticated': 'Local Users',
 'expires_in': 1800,
 'token': 'bf75b888-d08a-49b2-a714-01bed8647e6e',
 'user': {'allow_dialback': False,
          'break_seq': '\\x1bB',
          'clear_ports': '1-32,U1,U2',
          'data_ports': '1-32,U1,U2',
          'dialback_number': 'null',
          'escape_seq': '\\x1bA',
          'group': 'Administrators',
          'listen_ports': '1-32,U1,U2',
          'permissions': 'ad,nt,sv,dt,lu,ra,um,dp,ub,rs,fc,dr,sn,wb,sk,po,do,md,rp,sw',
          'power_outlets': '1-8',
          'uid': 0,
          'username': 'sysadmin'}}


### List active API sessions

In [6]:
get_active_sessions = sessions()

if get_active_sessions is not None:
    data = get_active_sessions.json()
    results = data.get("results", [])
    sorted_results = sorted(results, key=lambda s: s["login_time"])

    print(
        f"{'ID':<8} {'Type':<14} {'Username':<16} {'Remote IP':<20} {'Login Time'}"
    )
    print("-" * 72)
    for s in sorted_results:
        print(
            f"{str(s.get('id', '')):<8}"
            f"{s.get('session_type', ''):<14}"
            f"{s.get('username', ''):<16}"
            f"{s.get('remote_ip', ''):<20}"
            f"{s.get('login_time', '')}"
        )

https://10.40.21.41/api/v2/sessions successful (200)
ID       Type           Username         Remote IP            Login Time
------------------------------------------------------------------------
4865    REST API      sysadmin        10.40.21.223        2026-06-26T11:22:19Z
4869    REST API      sysadmin        10.40.21.223        2026-06-26T11:23:32Z


### Check current firmware boot bank

In [7]:
get_firmware_bootbank = firmware_bootbank()

if get_firmware_bootbank is not None:
    pprint(get_firmware_bootbank.json())

https://10.40.21.41/api/v2/firmware/bootbank successful (200)
{'bank': 1}


### Inspect firmware update log

In [8]:
get_firmware_log = firmware_log()

if get_firmware_log is not None:
    print(get_firmware_log.json().get("logs", ""))

https://10.40.21.41/api/v2/firmware/log successful (200)
Update Start: 06/20/26 14:28
Current Bank Firmware Version: 9.7.0.0R18
Alternate Bank Firmware Version: 9.7.0.0R17
Initiating firmware update via Percepxion with slc9update-9.7.0.0R18.tgz.
Saving current configuration to before_062026_1428-slc9cfg.tgz.
Running slc9update-9.7.0.0R18.tgz ROM firmware update.
Checking firmware version dependencies...
Current running firmware version is 9.7.0.0, update version is 9.7.0.0...
Current running firmware version 9.7.0.0 meets minimum firmware requirements.
Firmware upgrade started
bootbank_kcmdline: 'Dual Bank 2'  bootbank_environment: 'Dual Bank 2'
image 1 type: gzipped ext4 rootfs image
image 1 version: 9.7.0.0R18  (current: 9.7.0.0R17)
image 1 product code: SE  (current: SE)
writing image to partition 0 () size=0x8782dd0...
0+47996 records in
0+47996 records out
1572864000 bytes (1.6 GB, 1.5 GiB) copied, 35.1577 s, 44.7 MB/s
image 2 type: www_tar_gz
image 2 version: 9.7.0.0R18  (current

### Check for available firmware updates

In [9]:
get_firmware_check = firmware_check()

if get_firmware_check is not None:
    pprint(get_firmware_check.json())

https://10.40.21.41/api/v2/firmware/check successful (200)
{'latest_version': '9.8.0.0R7',
 'latest_version_notes': 'https://update.lantronix.com/SLC9000/released/SLC9000_v9.8.0.0_relnotes.txt',
 'latest_version_url': 'https://update.lantronix.com/SLC9000/released/slc9update-9.8.0.0R7.tgz',
 'up_to_date': False}


### Retrieve system software versions

In [10]:
get_system_version = system_version()

if get_system_version is not None:
    pprint(get_system_version.json())

https://10.40.21.41/api/v2/system/version successful (200)
{'bootloader_version': '2.0.0.0R12',
 'current_firmware_version': '9.7.0.0R18',
 'ec_version': '2.1',
 'io_module_revisions': '16SPF, 16UBB, 16ESB',
 'io_module_types': 'RJ45-16, USB-16, ETH-16',
 'main_board_version': 'unknown',
 'model': 'SLC9032',
 'power_supplies': 'AC, 2 power supplies',
 'sw_dnsmasq_version': '2.90',
 'sw_expect_version': '5.45.4',
 'sw_kernel_version': '6.6.52',
 'sw_ldap_version': '153',
 'sw_ntp_version': '4.2.8p18@1.4062-o',
 'sw_python_version': '3.13.2',
 'sw_radius_version': '3.0.0',
 'sw_rip_version': '9.1.3',
 'sw_ssh_version': 'OpenSSH_10.0p2, OpenSSL 3.4.1 11 Feb 2025',
 'sw_syslog_version': '2.7.1',
 'sw_tacacs_version': '1.6.0',
 'sw_tcl_version': '8.6',
 'sw_telnet_version': 'netkit-telnet-0.17',
 'sw_tls_version': 'OpenSSL 3.4.1 11 Feb 2025 (Library: OpenSSL 3.4.1 11 Feb '
                   '2025)',
 'sw_ttyd_version': '1.7.7',
 'sw_vpn_version': 'strongSwan U5.9.14/K6.6.52',
 'sw_webserve

### Check Zero Touch Provisioning (ZTP) status

In [11]:
get_system_ztp = system_ztp()

if get_system_ztp is not None:
    data = get_system_ztp.json()

    print(f"ZTP Status:      {data.get('status')}")
    print(f"DHCP Status:     {data.get('dhcp_status')}")
    print(f"Config Status:   {data.get('config_status')}")
    print(f"Firmware Status: {data.get('firmware_status')}")
    print(f"Last Operation:  {data.get('last_operation')}")

    # Show only Percepxion-relevant log entries
    px_lines = [
        l for l in data.get('log', [])
        if l.startswith('PX:') or l.startswith('ZTP &')
    ]
    if px_lines:
        print("\nPercepxion events:")
        for line in px_lines:
            print(f"  {line}")

https://10.40.21.41/api/v2/system/ztp successful (200)
ZTP Status:      Not_Run
DHCP Status:     Success
Config Status:   Not_Attempted
Firmware Status: Not_Required
Last Operation:  2026-06-08T13:07:33.200000Z

Percepxion events:
  ZTP & Bootstrap Onboarding Start: 06/06/26 20:04
  ZTP & Bootstrap Onboarding Start: 06/06/26 20:13
  PX: Registered to cloud percepxion.ai at 06/06/26 20:14
  PX: Checking for configuration update at 06/06/26 20:14
  PX: Checking for firmware update at 06/06/26 20:14
  PX: Registered to cloud gopercepxion.ai at 06/08/26 13:07
  PX: Checking for configuration update at 06/08/26 13:07
  PX: Checking for firmware update at 06/08/26 13:07


### Detect Percepxion cloud endpoint

The SLC9000 reports which Percepxion instance it is registered to inside the ZTP log. Detecting it here rather than hardcoding means the notebook works whether the device is on `gopercepxion.ai` (sandbox/TSE) or `percepxion.ai` (production). To override, set `PERCEPXION_URL` in `config.json`.

In [12]:
# Percepxion environment toggle
# percepxion.ai = production (Cisco Live tenant)
# gopercepxion.ai = sandbox/TSE
# Set PERCEPXION_URL in config.json to force a specific environment.
# Leave PERCEPXION_URL empty to auto-detect from this device's ZTP log.

import re

# Auto-detect from ZTP log; config.json PERCEPXION_URL takes precedence if set
if not PERCEPXION_HOST and get_system_ztp is not None:
    for line in get_system_ztp.json().get('log', []):
        m = re.search(r'Registered to cloud (\S+) at', line)
        if m:
            PERCEPXION_HOST = m.group(1)
            break

if PERCEPXION_HOST:
    print(f'Percepxion cloud:    {PERCEPXION_HOST}')
    print(f'Device dashboard:    https://{PERCEPXION_HOST}')
else:
    print('Percepxion host not detected, set PERCEPXION_URL in config.json if needed')

Percepxion cloud:    api.gopercepxion.ai
Device dashboard:    https://api.gopercepxion.ai


### Retrieve baseline configuration

Pulls the CLI command set that Percepxion applied to this device during ZTP enrollment. This closes the ZTP story: the device called home, Percepxion recognized it, and pushed exactly these commands as the managed baseline.

In [13]:
baseline_response = config_baseline()

if baseline_response is not None:
    data = baseline_response.json()
    commands = data.get('commands', data.get('config', []))

    if isinstance(commands, list):
        print(f'Baseline config, {len(commands)} commands:\n')
        for cmd in commands[:25]:
            print(f'  {cmd}')
        if len(commands) > 25:
            print(f'  ... ({len(commands) - 25} more)')
    elif commands:
        print(commands)
    else:
        print('No baseline config on this device (ZTP config push not triggered).')

NameError: name 'config_baseline' is not defined

### Monitor system health and hardware status

In [14]:
get_system_status = system_status()

if get_system_status is not None:
    pprint(get_system_status.json())

https://10.40.21.41/api/v2/system/status successful (200)
{'cell_link': 'null',
 'console_port': 'Connected',
 'eth1_link': 'Up',
 'eth2_link': 'Up',
 'eth3_link': 'Down',
 'eth4_link': 'Down',
 'ps1': 'Failed',
 'ps2': 'Ok',
 'temperature': 54,
 'uptime': 82063,
 'warranty_end_date': 'Jan 1, 2000'}


### Show device identity and metadata

In [15]:
get_system_identity = system_identity()

if get_system_identity is not None:
    pprint(get_system_identity.json())

https://10.40.21.41/api/v2/system/identity successful (200)
{'device_id': '00204ADCOZG6CHHBENW6GWLKXKA58G05',
 'hostname': 'slc9000-dg-02',
 'rack': '1',
 'rack_cluster': '1',
 'rack_row': '1',
 'serial_number': '000F2C030AE0',
 'site_tag': ''}


### Display network interface configuration

In [16]:
get_network_interfaces = network_interfaces()

if get_network_interfaces is not None:
    pprint(get_network_interfaces.json())

https://10.40.21.41/api/v2/network/interfaces successful (200)
{'dns_1': '10.40.21.1',
 'eth1_ipv4': '10.40.21.41',
 'eth1_ipv6': 'fdbc:284b:d3c0:72d5:020f:2cff:fe03:0ae0/64',
 'eth1_link': 'Up',
 'eth1_mask': '255.255.255.0',
 'eth2_ipv4': '192.168.1.102',
 'eth2_ipv6': 'fda3:2473:ff62:0000:0200:00ff:fe00:0002/64',
 'eth2_link': 'Up',
 'eth2_mask': '255.255.255.0',
 'eth3_ipv4': '',
 'eth3_ipv6': '',
 'eth3_link': 'Down',
 'eth3_mask': '',
 'eth4_ipv4': '',
 'eth4_ipv6': '',
 'eth4_link': 'Down',
 'eth4_mask': '',
 'gateway_ipv4': '10.40.21.1',
 'gateway_ipv6': '',
 'ntp_server_1': None}


## Serial Port Inventory and Managed Devices

The SLC9000 exposes all 32 serial ports and any attached managed devices via the REST API. These three endpoints are the core of the OOB value proposition: port inventory, active connections, and automatically discovered network equipment.

### Add serial port and managed device helper functions

In [17]:
# Serial Ports
def ports(status_filter=None):
    """
    Get all serial port statuses.
    Optional status_filter: "SSH", "Managed", "Idle", etc.
    """
    path = "/api/v2/ports"
    if status_filter:
        path += f"?status_filter={status_filter}"
    return api_call("GET", path)

def port_status(port_id):
    """
    Get status for a single port by number (1-32) or name.
    Includes HW signals, byte counters, errors, and attached managed device.
    """
    return api_call("GET", f"/api/v2/ports/{port_id}/status")

# Connections
def connections():
    """
    Get all active IP and direct connections to serial ports.
    """
    return api_call("GET", "/api/v2/connections")

# Managed Devices
def managed_devices(connection_filter=None):
    """
    Get manually added or auto-discovered managed devices.
    Optional connection_filter: "serial" or "ethernet"
    """
    path = "/api/v2/managed_devices"
    if connection_filter:
        path += f"?connection_filter={connection_filter}"
    return api_call("GET", path)

def managed_device_status(device_id):
    """
    Get status for a single managed device by name, serial port number,
    or switch port number. Returns model, OS version, hostname, management
    IP, temperature, CPU, memory, and uptime.
    """
    return api_call("GET", f"/api/v2/managed_devices/{device_id}/status")

# Config
def config_baseline():
    """
    Get the baseline configuration applied by Percepxion during ZTP.
    Returns the set of CLI commands that constitute the managed baseline.
    """
    return api_call("GET", "/api/v2/config/baseline")

### Inventory all 32 serial ports

Every serial port is addressable via the API, physical type (RJ45 or USB), current status, and any attached managed device. The `status_filter` query parameter narrows the response to ports whose status contains a given string, e.g. `"Managed"` or `"SSH"`.

In [18]:
ports_response = ports()

if ports_response is not None:
    data = ports_response.json()
    total = data.get("total_ports", 0)
    port_list = data.get("ports", [])

    print(f"Total serial ports: {total}\n")
    print(
        f"{'ID':<5} {'Name':<20} {'Type':<6} {'Status':<30} {'Managed Device'}"
    )
    print("-" * 80)
    for p in port_list:
        md = ""
        if p.get("md_name"):
            md = p["md_name"]
            if p.get("md_type"):
                md += f" ({p['md_type']})"
        print(
            f"{str(p.get('id', '')):<5}"
            f"{p.get('name', ''):<20}"
            f"{p.get('type', ''):<6}"
            f"{p.get('status', ''):<30}"
            f"{md}"
        )

https://10.40.21.41/api/v2/ports successful (200)
Total serial ports: 32

ID    Name                 Type   Status                         Managed Device
--------------------------------------------------------------------------------
1    MikroTik            rj45  Idle                          
2    C892FSP-K9          rj45  Idle                          
3    Port-03             rj45  Idle                          
4    Port-04             rj45  Idle                          
5    Port-05             rj45  Idle                          
6    Port-06             rj45  Idle                          
7    Port-07             rj45  Idle                          
8    Port-08             rj45  Idle                          
9    Port-09             rj45  Idle                          
10   Port-10             rj45  Idle                          
11   Port-11             rj45  Idle                          
12   Port-12             rj45  Idle                          
13   Port-13         

### Filter for managed ports only

Narrows the port list to only those currently under device management, the ports actively discovering and monitoring connected gear.

In [19]:
managed_ports_response = ports(status_filter="Managed")

if managed_ports_response is not None:
    data = managed_ports_response.json()
    port_list = data.get("ports", [])

    if not port_list:
        print("No ports currently in Managed status.")
    else:
        print(f"{len(port_list)} managed port(s):\n")
        for p in port_list:
            print(f"  Port {p['id']:>2}: {p['name']:<20} {p['status']}")
            if p.get("md_name"):
                print(
                    f"           Device: {p['md_name']}"
                    f" ({p.get('md_type', 'unknown type')})"
                )

https://10.40.21.41/api/v2/ports?status_filter=Managed successful (200)
32 managed port(s):

  Port  1: MikroTik             Idle
  Port  2: C892FSP-K9           Idle
  Port  3: Port-03              Idle
  Port  4: Port-04              Idle
  Port  5: Port-05              Idle
  Port  6: Port-06              Idle
  Port  7: Port-07              Idle
  Port  8: Port-08              Idle
  Port  9: Port-09              Idle
  Port 10: Port-10              Idle
  Port 11: Port-11              Idle
  Port 12: Port-12              Idle
  Port 13: Port-13              Idle
  Port 14: Port-14              Idle
  Port 15: Port-15              Idle
  Port 16: Port-16              Idle
  Port 17: Port-17              Idle
  Port 18: Port-18              Idle
  Port 19: Port-19              Idle
  Port 20: Port-20              Idle
  Port 21: Port-21              Idle
  Port 22: Port-22              Idle
  Port 23: Port-23              Idle
  Port 24: Port-24              Idle
  Port 25: Port-25 

### Active connections to serial ports

Lists every active IP or direct connection currently open to a device port, who is connected, from where, for how long, and whether they authenticated. In a real OOB incident, this shows which engineers are actively working the problem.

In [20]:
connections_response = connections()

if connections_response is not None:
    data = connections_response.json()
    conn_list = data.get("list", [])

    if not conn_list:
        print("No active connections.")
    else:
        print(f"{len(conn_list)} active connection(s):\n")
        for c in conn_list:
            duration_min = c.get("duration", 0) // 60
            print(f"  [{c.get('id')}] {c.get('description')}")
            print(
                f"       Source: {c.get('source_ip', 'N/A')}"
                f"  User: {c.get('username', 'N/A')}"
            )
            print(
                f"       Duration: {duration_min}m"
                f"  Status: {c.get('status')}"
            )
            print()

https://10.40.21.41/api/v2/connections successful (200)
1 active connection(s):

  [2] Console Port to Command Line
       Source:   User: 
       Duration: 1367m  Status: connected (waiting)



### Managed devices discovered on serial ports

The SLC9000 fingerprints and discovers network equipment connected to its serial ports, capturing hostname, model, OS version, management IP, and live health telemetry. This is the AOOB value: the console server knows *what* it's managing, not just which port it's on.

In [21]:
managed_response = managed_devices(connection_filter="serial")

if managed_response is not None:
    data = managed_response.json()
    total = data.get("total_devices", 0)
    device_list = data.get("devices", [])

    print(f"Serial managed devices discovered: {total}")

    for d in device_list:
        print(f"\n  Port {d.get('port_id', '?')}, {d.get('name', 'unnamed')}")
        print(f"    Type:       {d.get('type', 'N/A')}")
        print(f"    Model:      {d.get('model', 'N/A')}")
        print(f"    Hostname:   {d.get('hostname', 'N/A')}")
        print(f"    OS Version: {d.get('os_version', 'N/A')}")
        print(f"    Mgmt IP:    {d.get('management_ipv4', 'N/A')}")
        temp = d.get("temperature")
        if temp is not None:
            print(
                f"    Health:     {temp}°C"
                f"  CPU {d.get('cpu_usage', '?')}%"
                f"  Mem {d.get('memory_usage', '?')}%"
            )

https://10.40.21.41/api/v2/managed_devices?connection_filter=serial successful (200)
Serial managed devices discovered: 1

  Port 2, c892fsp-k9-02
    Type:       cisco ios
    Model:      C892FSP-K9
    Hostname:   c892fsp-k9-02
    OS Version: 15.7(3)M
    Mgmt IP:    10.40.21.46
    Health:     0°C  CPU 2%  Mem 0%


### Single port detail, hardware signals and byte counters

The per-port status endpoint returns hardware signal states (CTS, DSR, DTR, RTS) and byte counters alongside the status summary. Change `DEMO_PORT` to inspect any port by number or name.

In [22]:
DEMO_PORT = 1  # change to any port number (1-32) or port name

port_detail = port_status(DEMO_PORT)

if port_detail is not None:
    p = port_detail.json()
    print(f"Port {p.get('id')}: {p.get('name')}  [{p.get('type', '').upper()}]")
    print()
    print(
        f"  HW Signals:   CTS={str(p.get('cts', '?')):<6}"
        f"DSR={str(p.get('dsr', '?')):<6}"
        f"DTR={str(p.get('dtr', '?')):<6}"
        f"RTS={str(p.get('rts', '?'))}"
    )
    print(f"  Bytes in/out: {p.get('bytes_input', 0):,} / {p.get('bytes_output', 0):,}")
    print(f"  Errors:       {p.get('errors', 0)}")
    print(f"  Status:       {p.get('status')}")
    if p.get('md_name'):
        print(f"  Managed:      {p['md_name']} ({p.get('md_type', 'unknown type')})")

https://10.40.21.41/api/v2/ports/1/status successful (200)
Port 1: MikroTik  [RJ45]

  HW Signals:   CTS=True  DSR=True  DTR=True  RTS=True
  Bytes in/out: 1,599 / 112
  Errors:       0
  Status:       Idle


### Fleet health dashboard, all managed devices

Iterates every serial-attached managed device and calls the per-device status endpoint to build a live health table. This is the AOOB value in one view: every connected device, its OS version, and its real-time health, all via the SLC9000 REST API.

In [23]:
managed_response = managed_devices(connection_filter='serial')

if managed_response is not None:
    device_list = managed_response.json().get('devices', [])

    if not device_list:
        print('No serial managed devices found.')
    else:
        print(
            f"{'Port':<6} {'Device':<20} {'Model':<22}"
            f" {'OS Version':<18} {'Mgmt IP':<18} {'CPU%':<6} {'Mem%':<6} Temp°C"
        )
        print('-' * 106)
        for d in device_list:
            det = managed_device_status(d.get('port_id', d.get('name', '')))
            if det is None:
                continue
            v = det.json()
            temp = v.get('temperature')
            temp_str = f'{temp}°C' if temp is not None else 'N/A'
            print(
                f"{str(v.get('port_id', '')):<6}"
                f"{v.get('name', ''):<20}"
                f"{v.get('model', 'N/A'):<22}"
                f"{v.get('os_version', 'N/A'):<18}"
                f"{v.get('management_ipv4', 'N/A'):<18}"
                f"{str(v.get('cpu_usage', '?')):<6}"
                f"{str(v.get('memory_usage', '?')):<6}"
                f"{temp_str}"
            )

https://10.40.21.41/api/v2/managed_devices?connection_filter=serial successful (200)
Port   Device               Model                  OS Version         Mgmt IP            CPU%   Mem%   Temp°C
----------------------------------------------------------------------------------------------------------
https://10.40.21.41/api/v2/managed_devices/2/status successful (200)
                          N/A                   N/A               N/A               ?     ?     N/A


### Log out and terminate API session

In [24]:
logout = user_logout()

if logout is not None:
    pprint(logout.json())

https://10.40.21.41/api/v2/user/login successful (200)
{'code': 'SUCCESS', 'message': ['API session terminated'], 'status': 200}


### Other available calls

> **Caution:** The cells below are not part of the standard demo flow. `config_commands` retrieves the full running config. `system_reboot` immediately reboots the device, do not run it during a demo unless you intend to demonstrate the reboot and recovery flow.

In [ ]:
get_config_commands = config_commands()

if get_config_commands is not None:
    pprint(get_config_commands.json())

> **Caution:** Running the cell below reboots the device immediately. Do not execute during a demo unless you intend to show the reboot and recovery sequence.

In [ ]:
post_system_reboot = system_reboot()

if post_system_reboot is not None:
    pprint(post_system_reboot.json())